# Parameter Hierarchy

- Account Parameter
- Session Parameter
- Object Parameter

- Account Parameter : high privilege, set parameter at account level only
- Session Parameter : Administators with approproate prilege can set up session parameters using CREATE USER, ALTER USER. Whatever value we setup in session parameter, it becomes default to all users.
- Object Parameter : the values set becomes default to Warehouse, database, user.

# Parameter Precedence

In Snowflake, paramters have a clear hierarchy where settings at a more specific level override those at a broader level. The order of parameter precednece, from highest to lowest is

- Default value(Sys level ): Snowflake provides default vakue for every parameter
- Account level : Account administrator can set a default value for the entire account using ALTER ACCOUNT. This value overrides the Snowflake default.
- Object level : Parameters can be set for specific objetcs like warehousem, database, schemas, etc using commands like ALTER WAREHOUSE, ALTER DATABASE.. This overrieds account level settigns for that specific object.
- User level :Administrators or user can set paramters defaukt for a speicific user using alter user  This value overrides the objects and account setting for that user.
- Session level- Within an active session, a user can explicitly set parameters using ALTER SESSION.
- Statement/ Query level - certian parameters like statement_timeout_in_seconds can be applied to an individual SQL statement or passed as arguments in functions/methods, which takes the highest precednece for that specific operation

# Warehouse

It is a cluster of compute resources in Snowflake. A virtual warehouse are of two types:
- Standard
- Snowpark Optimized

Snowpark Optimized warehouses let you configure the available memory resources and CPU architechture on a single node instance for your workloads.
- default config : 16x memory per node compared to standard warehouse
- we can optionally configure additional memory per node and specify CPU architechture using the resource_constraint_property using CREATE WAREHOUSE, ALTER WAREHOUSE

| Memory upto | CPU arch | Resource Constraint | Min Warehouse size |
|---|---|---|---|
|16Gb | default or x86 | Memory_1x, Memory_1x_x86| XS|
| 256 Gb | default or x86| Memory16x,Memory16x_x86| M |
| 1 TB | default or x86 | Memory_64x,Memory64x_x86| L|

Standard Warehouse
- Gen1 : It represents Snowflake's original industry leading standard virtual warehouse
- Gen2 : It represents the next generation of Snowflake's standard virtual warehouse

| Gen1 | Gen2|
|---|---|
| Ratio of CPU to Memory is fixed | Swaps out virtual machines for intances that have more memory and faster SSds|
| Optimized for Row/Columnar sql| Optimized for memory heavy objects |
| high risk of remote spilling| designed to eliminate remote spilling |
| great for small queries | great for fewer massive complex task |
| 1 credit per hour | 1.5 - 2 credits per hour|

- You can change the warehouse from gen1 to gen2 and vice-versa even when the warehouse is running or suspended.

# Scaling Policies

is a feature of multi-cluster warehouses runing in auto scale mode. These policies determine how quickly Snowflake adds or removes clusters to manage <font color = red> fluctuating query loads</font> and <font color = red>balance performace with cost</font>.

- Standard : Prioritize performance by quickly starting new clusters when queries are queued or current reousrces are insufficient. Suitable for workloads like BI dashboards. New clusters are added quickly within 20 secs wait between adding additional clusters for warehouses with 10 or few max clusters. Clusters are shut down once the queries are complete

- Economy
Prioritize cost by aiming to keep existing clusters busy before adding new clusters, potentially leading to longer query queues. Recommended for non urgent tasks like ETL/ELT jobs. New cluster is started if there is enough estimated load for for than 6 mins

ALTER WAREHOUSE SET SCALING_POLICY = 'ECONOMY';
OR 
ALTER WAREHOUSE SET SCALING_POLICY='STANDARD';

- USE CASES

1. AD HOC QUERIES: 
Goal: Balance performace for unpredictable highly unique queries
Warehouse: Standard
Size: Start with XS or S
Config : Auto Suspend (60 sec), Scaling Policy: Standard, Multi Cluster
Why: Ad Hoc queries are rarely repititive. Keeping a warm cache is not important becuase uusers query differnet data each time. so short auto suspend are ideal.

2. Data Loading (ETL/ELT)
Goal: High throughput and stability for batch processing
Warehouse: Standard for typical SQL command and Snowpark Optimized if you ETL involves heavy queries that spills to disk.
Size: Medium to Large
Config: Auto Suspend (0-60sec), Scaling Policy: Economy, MultiCLuster : Usually 1 cluster. ETL is typically managed by scheduler.
Why: Data loading is heavy lifting task. You want raw power to handle large files efficiently.

3. BI Reporting
Goal : Low Latency and High concurrency for many simulatanoeusly dashboard users.
Warehouse: Standard
Size: S to M
Conf: Auto Suspend: 10 mins : Crucial to keep data cache warm, Scaling: Standard, multi lcuster: 3-10
Why: BI tools like power BI send many small, concurrent queries. YOu need horizantal scaling(Mlti cluster to prevent queing)

| Use case | Type | Size | Auto suspend | Scaling Policy |
|---|---|---|---|---|
| Ad Hoc | Standard | XS - S | 60 sec | Standard |
| Data Load | Standard for simple SQl/ Optimized for disk spill queries | M - XL | 0 - 60 s | Economy |
| Bi | Standard | S- M | 10 mns | Standard |


# Sizing
- Vertical Scaling ( Scale up / down)
 It refers to changing the size of warehouse. Each size up increase the compute power and cost.
 Best practice: Start with XS and look Query history for spilling
 Scale up: do it for complex queries.
 Scale down: if warehouse load monitoring has white spaces(idle time)
 Note: resizing can be done with zero downtime even while queries are running

 - Horizantal Scaling ( Out / In)
 It refers to adding more clusters for the same size.
 Bes Practice: Scale Out for high concurrency. If you have more than 100 users accessing the BI dashboard at 9 AM, you need more clusters rather than increasing the size of warehouse
 Scaling: Standard: adds clusters whenevr queries queue
 Scaling : Economy: only adds clusters if there are sure they will be fully utilized for 6 mins.

 - AuTO SUSPEND
 Best Practice : 60 s for ETL and Adhoc and 10 mns for BI tasks
 - Auto Resume: Always keep this on. There is almost no reason to manage warehouse starts.

 # Micro Partitions
 All data are loaded into partitions called micro partitions. Each partition contains 50 - 500 mB of uncompressed data, organized in columnar fashion

 Metadata :
 -  range of values for each column in the micro partition
 - no of distinct values
 - additional properties for optimization and query processing.

 Micro partitionins are  automatically perofmred on all snowflake tables and are transparently partitioned using the ordering of the data as it is inserted / laoded. This is known as natural clustering.


 -- Table CLustering

 process of physically organzing table within its micropartitions to align with specific column values

 The problem: Over time as data is deleted, updated or merged, the natural order breaks. This might lead data skewness where a specific range of data is scattered across thousands of micro partitions.

 Sol: Clustering key: one or more columns that snowflake uses to corlocate data
 when to use:
 - multi TBs of data
 - query performace degrades
 - clustering depth is high

 # Snowflake Notebooks

 unified deelopment interface in snowsight that offers interactive, cell based programmignenv. You can leverage your snowflake data to perform exploratory data analysis and develop machine learning models, and data engineering workflows all within the same interface.

 # Snowfalke stearmlit

 fully managed service that allows developers to securely build deploy and share interactive data applications using pure python within the snowflake data cloud.

 # SNowpark

 developer framework for snowflake that allows to write code in non sql language specifically, python, java scala.

 # Snowflake cortex
 built in AI and LLM directly within Snowflake

 Layers
 - Agents & insights : Agent apps, cortex agents, cortex ai functions
 - Processing and retrieval: cortex search, cortex analyst, document ai
 - models and governance: open ai, anthropic/ RBAC, guardrails
 - data: Snowflake marketplace, openflow, external

